# 📘 Introduction

With the rapid growth of user-generated content, analyzing large volumes of textual data has become an essential task in modern machine learning. One of the most common applications is **sentiment analysis**, where the goal is to automatically determine the opinion or rating expressed in a piece of text, such as product reviews.

Traditional machine learning approaches rely on handcrafted features and simpler models, but they often struggle to capture the complexity and nuance of natural language. Recent advances in transformer-based large language models, such as Qwen2.5-0.5B, have significantly improved performance by learning deep contextual representations of text.

However, fully fine-tuning such large models is computationally expensive and requires substantial memory and training time. To address this challenge, modern techniques like **LoRA** enable efficient adaptation by training only a small number of additional parameters while keeping the majority of the pretrained model fixed.

In this project, a pretrained transformer model is adapted for **multi-class sentiment classification** on Amazon book reviews. The task involves predicting a rating from 1 to 5 stars based on the textual content of each review. The pipeline includes dataset preparation, tokenization, efficient fine-tuning using LoRA, training with optimized hyperparameters, and evaluation using metrics such as accuracy and mean absolute error.

This work demonstrates how large language models can be effectively repurposed for domain-specific tasks using parameter-efficient techniques, making them practical even in constrained environments such as Kaggle notebooks.


## 📦 Environment Setup & Authentication

This section prepares the notebook environment by installing required libraries, importing dependencies, authenticating with Hugging Face, and configuring the compute device.

---

### 🔧 Library Installation

We install the core libraries required for the training pipeline:

* **transformers** — provides pretrained models and tokenizers
* **datasets** — efficient dataset loading and processing
* **peft** — enables LoRA-based fine-tuning
* **accelerate** — handles efficient training on hardware
* **scikit-learn** — evaluation metrics (accuracy, confusion matrix)
* **matplotlib** — visualization support

> Note: PyTorch is already pre-installed in Kaggle environments, so it is not reinstalled.

---

### 📚 Imports

The notebook imports modules for:

* Deep learning and tensor operations (PyTorch)
* Model loading and tokenization (Transformers)
* Dataset handling (Datasets + DataLoader)
* Optimization and training utilities
* LoRA-based parameter-efficient fine-tuning

---

### 🔐 Hugging Face Authentication

The notebook authenticates using a Hugging Face token stored securely in Kaggle Secrets.

This allows:

* Downloading models and datasets
* Saving and pushing checkpoints
* Accessing private repositories

---

### ⚡ Device Configuration

The system checks whether a GPU is available:

* If available → training runs on **CUDA (GPU)**
* Otherwise → falls back to **CPU**

Using a GPU significantly speeds up training, especially for large transformer models.

---

### ✅ Outcome of This Cell

After this step, the environment is fully ready with:

* Required libraries installed
* All dependencies imported
* Hugging Face access configured
* Compute device selected

---


In [ ]:
# ── Cell 1: pip installs + imports ──────────────────────────────────────────
# torch/torchvision are pre-installed on Kaggle — do NOT reinstall them.
# peft requires accelerate; install both together.
%pip install -q --no-deps datasets transformers huggingface_hub peft accelerate
%pip install -q --no-deps ipywidgets matplotlib scikit-learn

import os, math, glob as _glob, shutil
from itertools import islice

import torch
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset as hf_load
import datasets as _ds
from huggingface_hub import login

# ── HuggingFace auth: Kaggle Secrets or .env ────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient
    _hf_token = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass
    _hf_token = os.environ.get('HF_TOKEN')
    if not _hf_token:
        raise RuntimeError('HF_TOKEN not found in .env or environment')
login(token=_hf_token)
print('HuggingFace login: OK')

print('torch :', torch.__version__)
print('cuda  :', torch.cuda.is_available())
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

## ⚙️ Model & Training Configuration (Hyperparameters)

This section defines all key hyperparameters that control how the model is trained, how data is processed, and how LoRA fine-tuning is applied.

---

### 🤖 Model Configuration

* **Model:** Qwen2.5-0.5B
* **Task:** Multi-class sentiment classification
* **Number of labels:** 5 (corresponding to 1–5 star ratings)
* **Max sequence length:** 256 tokens

The model is adapted for classification by adding a prediction head that outputs probabilities over the 5 sentiment classes.

---

### 🔧 LoRA (Parameter-Efficient Fine-Tuning)

We use **LoRA** to fine-tune the model efficiently.

Instead of updating all parameters of the large model, LoRA:

* Freezes the original model weights
* Injects small trainable matrices into attention layers
* Learns task-specific adjustments with far fewer parameters

#### Key LoRA settings:

* **Rank (`r`) = 8** → controls size of adaptation matrices
* **Alpha = 16** → scaling factor for LoRA updates
* **Dropout = 0.1** → regularization to prevent overfitting
* **Target modules:**

  * `q_proj`, `k_proj`, `v_proj`, `o_proj`
    These correspond to **attention projections**, which are the most effective points for adaptation in transformer models.

---

### 📊 Training Configuration

These parameters control how learning happens:

* **Batch size = 32**

* **Gradient accumulation = 2**
  → Effective batch size = 64

* **Learning rate:**

  * Max LR = 2e-4
  * Min LR = 1e-5

> A higher learning rate is used because only LoRA parameters are being trained (not the full model).

* **Weight decay = 0.01**
  → Helps prevent overfitting by penalizing large weights

* **Epochs = 3**
  → Number of full passes through the dataset

* **Warmup ratio = 0.06**
  → Gradually increases learning rate at the start of training for stability

---

### 🎯 Why These Choices Matter

* LoRA reduces memory usage and training time significantly
* Higher learning rate speeds up adaptation without destabilizing the model
* Gradient accumulation allows larger effective batch sizes on limited GPU memory
* Warmup improves early training stability

---

### ✅ Outcome of This Cell

After this step, all core parameters for:

* Model behavior
* LoRA adaptation
* Training dynamics

are defined and ready to be used in the training pipeline.

---


In [ ]:
# ── Cell 2: hyperparameters ───────────────────────────────────────────────────
MODEL_NAME = 'Qwen/Qwen2.5-0.5B'
NUM_LABELS = 5
MAX_LENGTH = 256

# LoRA
LORA_R           = 8
LORA_ALPHA       = 16
LORA_DROPOUT     = 0.1
LORA_TARGET_MODS = ['q_proj', 'k_proj', 'v_proj', 'o_proj']

# Training
BATCH_SIZE         = 32
ACCUMULATION_STEPS = 2
MAX_LR             = 2e-4   # higher LR is normal for LoRA vs full fine-tune
MIN_LR             = 1e-5
WEIGHT_DECAY       = 0.01
NUM_EPOCHS         = 3
WARMUP_RATIO       = 0.06

## 🧠 Tokenizer, Model Initialization & LoRA Integration

This section loads the tokenizer and base model, configures them for classification, applies LoRA for efficient fine-tuning, and performs a sanity check to ensure everything works correctly.

---

### 🔤 Tokenizer Setup

We load the tokenizer associated with the pretrained model:

* Converts raw text into token IDs (numerical form)
* Applies padding and truncation to ensure uniform input size

#### Key configurations:

* **Padding token:** If missing, it is set to the end-of-sequence token
* **Padding side:** Right padding ensures compatibility with transformer models
* **Max length:** Controlled by `MAX_LENGTH` to limit sequence size

---

### 🤖 Model Loading

We load the pretrained model:

* Model: Qwen2.5-0.5B
* Task: Sequence classification with 5 output labels

#### Important details:

* The model is loaded in **bfloat16 precision**, which:

  * Reduces memory usage
  * Maintains numerical stability (same exponent range as FP32)
* No gradient scaling is needed due to bfloat16 stability

The model is also configured with the correct padding token ID.

---

### 🔧 Applying LoRA (Efficient Fine-Tuning)

We apply **LoRA** to the base model.

#### What LoRA does:

* Freezes the original pretrained weights
* Injects small trainable matrices into attention layers
* Enables efficient adaptation with minimal memory and compute

#### Configuration highlights:

* Targets attention projections: `q_proj`, `k_proj`, `v_proj`, `o_proj`
* Adds trainable adapters to these layers only
* Keeps the classification head (`score`) fully trainable and saved

This ensures:

* The model learns the new task effectively
* The final classification layer is properly updated

---

### 📊 Trainable Parameters Check

The model prints the number of trainable vs total parameters.

This confirms:

* Only a small subset of parameters (LoRA + classifier) are being trained
* The base model remains mostly frozen

---

### ⚡ Device Placement

The model is moved to the selected device:

* GPU (`cuda`) if available
* Otherwise CPU

---

### ✅ Sanity Check (Forward Pass)

A test input is passed through the model to verify correctness.

Steps:

1. Tokenize a sample sentence
2. Run a forward pass (no gradients)
3. Check output shape

Expected output:

```
(1, NUM_LABELS)
```

This confirms:

* Model is correctly configured for classification
* Output dimensions match the number of labels
* No runtime errors in forward propagation

---

### 🎯 Why This Step Matters

This is the **core setup stage** where:

* Raw text → tokens → model-ready inputs
* Base LLM → task-specific classifier
* LoRA adapters → enable efficient fine-tuning

If anything is wrong here, training will fail later — so the sanity check is critical.

---

### ✅ Outcome of This Cell

After this step:

* Tokenizer is ready
* Model is initialized and configured
* LoRA is successfully applied
* Model is placed on the correct device
* Forward pass is verified

---


In [ ]:
# ── Cell 3: tokenizer + model + LoRA ─────────────────────────────────────────
print(f'Loading tokenizer: {MODEL_NAME} ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print(f'Loading model: {MODEL_NAME} ...')
# Qwen2.5 config.json sets torch_dtype=bfloat16; load explicitly in bfloat16.
# BFloat16 has the same exponent range as FP32 so GradScaler is unnecessary —
# the Trainer uses plain autocast(dtype=torch.bfloat16) with no scaler.
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    dtype=torch.bfloat16,
)
base_model.config.pad_token_id = tokenizer.pad_token_id

# modules_to_save=['score'] ensures the classification head is fully
# fine-tuned and saved alongside the LoRA adapters.
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODS,
    modules_to_save=['score'],
    bias='none',
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()
model = model.to(DEVICE)

# ── sanity forward pass ───────────────────────────────────────────────────────
enc = tokenizer('This movie was fantastic!', return_tensors='pt',
                max_length=MAX_LENGTH, padding='max_length', truncation=True)
model.eval()
with torch.no_grad():
    out = model(enc['input_ids'].to(DEVICE), attention_mask=enc['attention_mask'].to(DEVICE))
assert out.logits.shape == (1, NUM_LABELS), f'Expected (1,{NUM_LABELS}), got {out.logits.shape}'
print('logits shape:', out.logits.shape)
print('forward pass OK')

## 📚 Dataset Preparation, Caching & DataLoaders

This section handles loading the dataset, caching it efficiently, preprocessing text into model inputs, and preparing batches for training and validation.

---

### 📦 Dataset Source

We use the Amazon reviews dataset:

* Dataset: Amazon Reviews 2023 Books Review
* Repository: `cogsci13/Amazon-Reviews-2023-Books-Review`
* Subset: Book reviews
* Total samples used: **1.5 million**

Each sample contains:

* Review text
* Star rating (1–5)

---

### ⚡ Dataset Caching Strategy

Loading 1.5M samples repeatedly is slow, so a caching system is used.

#### Workflow:

1. **Check local cache**

   * If dataset exists on disk → load instantly

2. **Check Hugging Face cache repo**

   * Downloads preprocessed dataset if available

3. **Fallback: Stream dataset**

   * Loads data progressively from source
   * Converts streamed data into a full dataset
   * Saves cache locally and optionally uploads to Hugging Face

This ensures:

* Faster reloads in future runs
* No repeated heavy downloads
* Efficient reuse across notebooks

---

### 🧠 Custom Dataset Class

A custom PyTorch `Dataset` is defined to process each sample.

#### Responsibilities:

* Extract review text and rating
* Convert rating → label (0 to 4)
* Tokenize text using the model tokenizer
* Return tensors required for training

#### Label mapping:

```id="labelmap"
1 star → 0  
2 star → 1  
3 star → 2  
4 star → 3  
5 star → 4  
```

Edge handling:

* Missing text → replaced with empty string
* Missing rating → defaulted to neutral (3-star)
* Labels are clamped to valid range

---

### 🔤 Tokenization

Each review is converted into:

* `input_ids` → token IDs
* `attention_mask` → indicates real tokens vs padding

#### Key settings:

* Fixed length = `MAX_LENGTH`
* Padding applied to shorter sequences
* Longer sequences are truncated

This ensures:

* Uniform input size for batching
* Compatibility with transformer models

---

### 🔀 Train–Validation Split

The dataset is split into:

* **Training set:** 90%
* **Validation set:** 10%

Using a fixed random seed ensures reproducibility.

---

### 📦 DataLoaders

PyTorch `DataLoader` is used to create batches.

#### Training loader:

* Shuffles data (important for learning)
* Batch size = defined earlier
* Uses multiple workers for faster loading

#### Validation loader:

* No shuffling (keeps evaluation consistent)

---

### ⚡ Performance Optimizations

* **`num_workers=2`** → parallel data loading
* **`pin_memory=True`** → faster CPU → GPU transfer
* **Batching** → efficient GPU utilization

---

### 🔍 Sanity Check

A sample batch is printed to verify:

* Correct tensor shapes
* Proper batching
* Expected keys:

  * `input_ids`
  * `attention_mask`
  * `label`

Example structure:

```id="batchshape"
{
  input_ids: (batch_size, max_length),
  attention_mask: (batch_size, max_length),
  label: (batch_size,)
}
```

---

### 🎯 Why This Step Matters

This is where raw data becomes **model-ready input**:

* Text → tokenized tensors
* Ratings → classification labels
* Dataset → efficient batches

If this step is wrong, training will silently fail or produce bad results.

---

### ✅ Outcome of This Cell

After this step:

* Dataset is loaded and cached
* Data is preprocessed and tokenized
* Train/validation splits are created
* DataLoaders are ready for training
* Batch structure is verified

---


In [ ]:
# ── Cell 4: AmazonReview Dataset + DataLoaders ───────────────────────────────
# Reuses the same HF Hub dataset cache as bert-finetune-kaggle.ipynb so no
# re-download is needed when both notebooks run in the same Kaggle account.

DATASET_REPO_ID          = 'cogsci13/Amazon-Reviews-2023-Books-Review'
DATASET_SPLIT            = 'full'
DATASET_LABEL            = 'books_review'
N_SAMPLES                = 1_500_000
HF_DATASET_CACHE_REPO_ID = 'AgentPhoenix7/SLM-project-dataset-cache'
CACHE_ROOT               = '/kaggle/working/data' if os.path.exists('/kaggle/working') else 'data'
DATASET_CACHE_NAME       = f'amazon_reviews_2023_{DATASET_LABEL}_{N_SAMPLES}'
DATASET_CACHE_DIR        = os.path.join(CACHE_ROOT, DATASET_CACHE_NAME)


class AmazonReview(Dataset):
    def __init__(self, hf_dataset, tokenizer, max_length=256):
        self.dataset    = hf_dataset
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item   = self.dataset[idx]
        text   = item.get('text') or ''
        rating = item.get('rating')
        rating = rating if rating is not None else 3
        label  = max(0, min(4, int(float(rating)) - 1))

        tokens = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        return {
            'input_ids':      tokens['input_ids'].squeeze(0),
            'attention_mask': tokens['attention_mask'].squeeze(0),
            'label':          torch.tensor(label, dtype=torch.long),
        }


def load_dataset_cache():
    if os.path.exists(DATASET_CACHE_DIR):
        print(f'Loading local dataset cache from {DATASET_CACHE_DIR}')
        return _ds.load_from_disk(DATASET_CACHE_DIR)

    if HF_DATASET_CACHE_REPO_ID:
        try:
            from huggingface_hub import snapshot_download
            print(f'Downloading dataset cache from HF Hub: {HF_DATASET_CACHE_REPO_ID}')
            os.makedirs(CACHE_ROOT, exist_ok=True)
            snapshot_download(
                repo_id=HF_DATASET_CACHE_REPO_ID,
                repo_type='dataset',
                allow_patterns=f'{DATASET_CACHE_NAME}/**',
                local_dir=CACHE_ROOT,
            )
            if os.path.exists(DATASET_CACHE_DIR):
                return _ds.load_from_disk(DATASET_CACHE_DIR)
            print('HF Hub dataset cache path not found; will stream source dataset.')
        except Exception as e:
            print(f'HF Hub dataset cache unavailable: {e}')
    return None


def save_dataset_cache(dataset):
    os.makedirs(CACHE_ROOT, exist_ok=True)
    tmp = DATASET_CACHE_DIR + '.tmp'
    if os.path.exists(tmp):
        shutil.rmtree(tmp)
    dataset.save_to_disk(tmp)
    os.rename(tmp, DATASET_CACHE_DIR)
    print(f'Saved local dataset cache to {DATASET_CACHE_DIR}')

    if HF_DATASET_CACHE_REPO_ID:
        try:
            from huggingface_hub import HfApi
            api = HfApi()
            api.create_repo(HF_DATASET_CACHE_REPO_ID, repo_type='dataset', exist_ok=True, private=True)
            api.upload_folder(
                folder_path=DATASET_CACHE_DIR,
                path_in_repo=DATASET_CACHE_NAME,
                repo_id=HF_DATASET_CACHE_REPO_ID,
                repo_type='dataset',
            )
            print(f'Uploaded dataset cache to HF Hub: {HF_DATASET_CACHE_REPO_ID}/{DATASET_CACHE_NAME}')
        except Exception as e:
            print(f'HF Hub dataset cache upload failed (non-fatal): {e}')


raw = load_dataset_cache()
if raw is None:
    print(f'Streaming {N_SAMPLES:,} samples from {DATASET_REPO_ID}/{DATASET_SPLIT} ...')
    stream = hf_load(DATASET_REPO_ID, split=DATASET_SPLIT, streaming=True)
    raw = _ds.Dataset.from_list(list(islice(stream, N_SAMPLES)))
    save_dataset_cache(raw)
print(f'Total samples : {len(raw):,}')
print('Columns       :', raw.column_names)

splits   = raw.train_test_split(test_size=0.1, seed=42)
train_ds = AmazonReview(splits['train'], tokenizer, max_length=MAX_LENGTH)
val_ds   = AmazonReview(splits['test'],  tokenizer, max_length=MAX_LENGTH)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)
val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)
print(f'Train batches : {len(train_loader):,}')
print(f'Val   batches : {len(val_loader):,}')

batch = next(iter(train_loader))
print('Sample batch  :', {k: tuple(v.shape) for k, v in batch.items()})

## 🏗️ Training Infrastructure (Scheduler, Optimizer, Trainer & Checkpointing)

This section defines the complete training engine — including how the model learns, how learning rates are adjusted, how validation is performed, and how checkpoints are saved and recovered.

---

### ⏱️ Checkpointing Strategy

* **Checkpoint frequency:** every 500 optimizer steps
* **Mid-epoch validation:** limited to ~6,400 samples for speed

This ensures:

* Progress is saved frequently
* Training can resume after interruptions (e.g., Kaggle session reset)
* Best-performing model is preserved

---

### 📉 Learning Rate Scheduler

A custom **cosine decay scheduler with warmup** is implemented.

#### Behavior:

1. **Warmup phase:**

   * Learning rate gradually increases from 0 → max LR
   * Stabilizes early training

2. **Cosine decay phase:**

   * Learning rate smoothly decreases to minimum LR
   * Prevents overshooting and improves convergence

This strategy is widely used in transformer training for stable and efficient optimization.

---

### ⚙️ Optimizer Configuration

The optimizer used is:

* **AdamW**

#### Key features:

* Handles adaptive learning rates
* Applies weight decay for regularization

#### Parameter grouping:

* **Decay group:** normal weights → weight decay applied
* **No-decay group:** biases & normalization layers → no decay

This improves generalization and prevents unnecessary regularization on sensitive parameters.

---

### 🧠 Trainer Class (Core Training Engine)

A custom `Trainer` class manages the entire training lifecycle.

---

#### 🔁 Training Loop (Per Batch)

For each batch:

1. Forward pass → model predicts logits
2. Loss computed using cross-entropy
3. Backpropagation computes gradients
4. Gradients accumulated over multiple steps
5. Optimizer updates parameters
6. Scheduler updates learning rate

---

#### 📦 Gradient Accumulation

* Accumulates gradients across multiple batches
* Simulates larger batch sizes without extra memory

Example:

* Batch size = 32
* Accumulation steps = 2
  → Effective batch size = 64

---

#### ⚡ Mixed Precision (bfloat16)

* Uses automatic casting to **bfloat16** for faster computation
* No gradient scaling needed (unlike float16)
* Improves speed while maintaining stability

---

#### ✂️ Gradient Clipping

* Limits gradient norm to prevent exploding gradients
* Ensures stable updates during training

---

### 📊 Validation

Validation is performed:

* Periodically during training
* At the end of each epoch

#### Metrics:

* **Validation loss**
* **Accuracy**

This helps:

* Track model performance
* Detect overfitting
* Select the best model

---

### 💾 Checkpointing System

Only **trainable parameters** are saved:

* LoRA adapters
* Classification head

This makes checkpoints:

* Very small (~10–30 MB instead of ~1 GB)
* Faster to save/load

#### Types of checkpoints:

* **Step checkpoints:** saved during training
* **Epoch checkpoints:** saved at epoch end
* **Best checkpoint:** lowest validation loss

---

### ☁️ Hugging Face Hub Integration

Checkpoints are optionally pushed to Hugging Face:

* Ensures persistence beyond Kaggle sessions
* Allows remote storage and reuse
* Keeps latest and best checkpoints synced

---

### 🔄 Resume Training

Training can resume from any checkpoint:

* Restores model weights
* Restores optimizer and scheduler state
* Continues from the correct step

This is critical for long training runs.

---

### 🧹 Checkpoint Cleanup

Old step checkpoints are pruned automatically to:

* Save disk space
* Keep only the most recent checkpoints

---

### 🎯 Why This Step Matters

This is the **core training system**:

* Controls how learning happens
* Ensures stability and efficiency
* Handles interruptions safely
* Tracks and preserves best performance

Without this, training would be unreliable and inefficient.

---

### ✅ Outcome of This Cell

After this step:

* Learning rate scheduling is defined
* Optimizer is configured
* Full training loop is implemented
* Validation system is ready
* Checkpointing and recovery are enabled
* Training can now run robustly end-to-end

---


In [ ]:
# ── Cell 5: training infrastructure ──────────────────────────────────────────
# CKPT_EVERY: how many optimizer steps between mid-epoch checkpoint + hub push.
# 1.5 M samples, batch 32, accum 2 → ~21 000 optimizer steps/epoch → 500 is fine.
CKPT_EVERY      = 500
MAX_VAL_BATCHES = 200   # cap mid-epoch validation to ~6 400 samples


class CosineScheduler:
    def __init__(self, optimizer, warmup_steps, total_steps, max_lr, min_lr):
        self.optimizer    = optimizer
        self.warmup_steps = warmup_steps
        self.total_steps  = total_steps
        self.max_lr       = max_lr
        self.min_lr       = min_lr
        self.current_step = 0

    def get_lr(self):
        if self.current_step >= self.total_steps:
            return self.min_lr
        if self.current_step < self.warmup_steps:
            return self.max_lr * self.current_step / self.warmup_steps
        progress = (self.current_step - self.warmup_steps) / max(1, self.total_steps - self.warmup_steps)
        return self.min_lr + 0.5 * (self.max_lr - self.min_lr) * (1 + math.cos(math.pi * progress))

    def step(self):
        for pg in self.optimizer.param_groups:
            pg['lr'] = self.get_lr()
        self.current_step += 1

    def state_dict(self):
        return {k: v for k, v in self.__dict__.items() if k != 'optimizer'}

    def load_state_dict(self, sd):
        for k, v in sd.items():
            setattr(self, k, v)


def configure_optimizer(model, lr, weight_decay):
    decay, no_decay = [], []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if 'bias' in name or 'norm' in name or 'LayerNorm' in name:
            no_decay.append(param)
        else:
            decay.append(param)
    return AdamW(
        [{'params': decay,    'weight_decay': weight_decay},
         {'params': no_decay, 'weight_decay': 0.0}],
        lr=lr, betas=(0.9, 0.999), eps=1e-8,
    )


class Trainer:
    def __init__(self, model, train_loader, val_loader, scheduler,
                 optimizer=None, device='cpu', use_amp=False,
                 accumulation_steps=1, checkpoint_dir='./checkpoints',
                 hf_repo_id=None):
        self.model              = model
        self.train_loader       = train_loader
        self.val_loader         = val_loader
        self.optimizer          = optimizer
        self.scheduler          = scheduler
        self.device             = device
        # Qwen2.5 is bfloat16; GradScaler is only for float16.
        # use_amp still enables autocast(dtype=torch.bfloat16) for speed.
        self.use_amp            = use_amp and device != 'cpu'
        self.accumulation_steps = accumulation_steps
        self.checkpoint_dir     = checkpoint_dir
        self.hf_repo_id         = hf_repo_id
        self.global_step        = 0
        self.best_val_loss      = float('inf')
        os.makedirs(checkpoint_dir, exist_ok=True)

        if not hf_repo_id:
            print('WARNING: hf_repo_id not set — checkpoints will not survive session end.')

        if torch.cuda.device_count() > 1:
            self.model = torch.nn.DataParallel(self.model)

    def _unwrap(self):
        return self.model.module if isinstance(self.model, torch.nn.DataParallel) else self.model

    # ── single epoch ─────────────────────────────────────────────────────────
    def train_epoch(self, epoch):
        self.model.train()
        total_loss = 0.0
        step = -1

        for step, batch in enumerate(self.train_loader):
            input_ids      = batch['input_ids'].to(self.device)
            attention_mask = batch['attention_mask'].to(self.device)
            labels         = batch['label'].to(self.device)

            if self.use_amp:
                with autocast(device_type='cuda', dtype=torch.bfloat16):
                    out  = self.model(input_ids, attention_mask=attention_mask)
                    loss = F.cross_entropy(out.logits, labels) / self.accumulation_steps
            else:
                out  = self.model(input_ids, attention_mask=attention_mask)
                loss = F.cross_entropy(out.logits, labels) / self.accumulation_steps
            loss.backward()

            if (step + 1) % self.accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                self.optimizer.step()
                self.optimizer.zero_grad()
                self.scheduler.step()
                self.global_step += 1

                if self.global_step % CKPT_EVERY == 0:
                    metrics = self.validate(max_batches=MAX_VAL_BATCHES)
                    self.model.train()
                    tag = f'step_{self.global_step}'
                    print(f'[step {self.global_step}]  '
                          f'val_loss={metrics["val_loss"]:.4f}  '
                          f'val_acc={metrics["accuracy"]:.4f}')
                    self.save_checkpoint(
                        epoch, metrics['val_loss'],
                        os.path.join(self.checkpoint_dir, f'{tag}.pt'),
                        checkpoint_type='step',
                    )
                    self._prune_step_checkpoints(keep=3)
                    if metrics['val_loss'] < self.best_val_loss:
                        self.best_val_loss = metrics['val_loss']
                        self.save_checkpoint(
                            epoch, metrics['val_loss'],
                            os.path.join(self.checkpoint_dir, 'best.pt'),
                            checkpoint_type='step',
                        )
                        print(f'  ** new best at step {self.global_step}: '
                              f'val_loss={self.best_val_loss:.4f} **')

            total_loss += loss.item() * self.accumulation_steps
            if step % 100 == 0:
                lr = self.optimizer.param_groups[0]['lr']
                print(f'  epoch {epoch}  step {step:>6d}  '
                      f'loss {loss.item() * self.accumulation_steps:.4f}  lr {lr:.2e}')

        if (step + 1) % self.accumulation_steps != 0:
            self.optimizer.zero_grad()
        return total_loss / max(1, len(self.train_loader))

    # ── validation ───────────────────────────────────────────────────────────
    def validate(self, max_batches=None):
        self.model.eval()
        total_loss, correct, total, n_batches = 0.0, 0, 0, 0
        limit = max_batches or len(self.val_loader)
        print(f'  [validate] running up to {limit} batches ...', flush=True)
        with torch.no_grad():
            for i, batch in enumerate(self.val_loader):
                if i >= limit:
                    break
                input_ids      = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels         = batch['label'].to(self.device)
                out        = self.model(input_ids, attention_mask=attention_mask)
                total_loss += F.cross_entropy(out.logits, labels).item()
                correct    += (out.logits.argmax(-1) == labels).sum().item()
                total      += labels.size(0)
                n_batches  += 1
        print('  [validate] done.', flush=True)
        return {'val_loss': total_loss / max(1, n_batches),
                'accuracy': correct / max(1, total)}

    # ── full training loop ────────────────────────────────────────────────────
    def train(self, num_epochs, start_epoch=0):
        for epoch in range(start_epoch, num_epochs):
            print(f'\n{"="*60}  Epoch {epoch+1}/{num_epochs}  {"="*60}')
            train_loss = self.train_epoch(epoch)
            metrics    = self.validate()
            self.model.train()
            print(f'  [epoch {epoch+1} end]  '
                  f'train_loss={train_loss:.4f}  '
                  f'val_loss={metrics["val_loss"]:.4f}  '
                  f'val_acc={metrics["accuracy"]:.4f}')
            self.save_checkpoint(
                epoch, metrics['val_loss'],
                os.path.join(self.checkpoint_dir, f'epoch_{epoch+1}.pt'),
                checkpoint_type='epoch',
            )
            if metrics['val_loss'] < self.best_val_loss:
                self.best_val_loss = metrics['val_loss']
                self.save_checkpoint(
                    epoch, metrics['val_loss'],
                    os.path.join(self.checkpoint_dir, 'best.pt'),
                    checkpoint_type='epoch',
                )
                print('  ** new best (epoch end) **')
        return self.best_val_loss

    # ── checkpointing ─────────────────────────────────────────────────────────
    # Only trainable params are saved (LoRA adapters + score head) — much
    # smaller than the full model state (~10–30 MB vs ~1 GB).
    def save_checkpoint(self, epoch, val_loss, path, checkpoint_type='step'):
        m = self._unwrap()
        trainable_sd = {
            k: v.detach().cpu()
            for k, v in m.named_parameters()
            if v.requires_grad
        }
        state = {
            'epoch':                epoch,
            'global_step':          self.global_step,
            'checkpoint_type':      checkpoint_type,
            'trainable_state_dict': trainable_sd,
            'optimizer_state_dict': self.optimizer.state_dict(),
            'scheduler_state_dict': self.scheduler.state_dict(),
            'val_loss':             val_loss,
            'best_val_loss':        self.best_val_loss,
        }
        torch.save(state, path)

        last_path = os.path.join(self.checkpoint_dir, 'last.pt')
        torch.save(state, last_path)

        if self.hf_repo_id:
            self._push_to_hub(last_path)
            if os.path.basename(path) == 'best.pt':
                self._push_to_hub(path, 'checkpoints/best.pt')

    def _push_to_hub(self, local_path, repo_path='checkpoints/last.pt'):
        try:
            from huggingface_hub import HfApi
            api = HfApi()
            api.create_repo(self.hf_repo_id, repo_type='model', exist_ok=True, private=True)
            api.upload_file(
                path_or_fileobj=local_path,
                path_in_repo=repo_path,
                repo_id=self.hf_repo_id,
                repo_type='model',
            )
            print(f'  [hub] ✓ {repo_path} → {self.hf_repo_id} '
                  f'(step {self.global_step})', flush=True)
        except Exception as e:
            print(f'  [hub] upload failed (non-fatal): {e}', flush=True)

    def _prune_step_checkpoints(self, keep=3):
        step_ckpts = sorted(_glob.glob(os.path.join(self.checkpoint_dir, 'step_*.pt')))
        for old in step_ckpts[:-keep]:
            os.remove(old)

    def load_checkpoint(self, path):
        ckpt = torch.load(path, map_location=self.device, weights_only=False)
        m = self._unwrap()
        current_sd = m.state_dict()
        for k, v in ckpt['trainable_state_dict'].items():
            if k in current_sd:
                current_sd[k].copy_(v.to(self.device))
        m.load_state_dict(current_sd)
        self.optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        self.scheduler.load_state_dict(ckpt['scheduler_state_dict'])
        self.global_step   = ckpt.get('global_step', 0)
        self.best_val_loss = ckpt.get('best_val_loss', float('inf'))
        ckpt_type = ckpt.get('checkpoint_type', 'epoch')
        print(f'Resumed from {os.path.basename(path)}  '
              f'(epoch={ckpt["epoch"]}, step={self.global_step}, '
              f'type={ckpt_type}, best_val_loss={self.best_val_loss:.4f})')
        return ckpt['epoch'] + (1 if ckpt_type == 'epoch' else 0)


print('Training infrastructure defined.')
print(f'CKPT_EVERY = {CKPT_EVERY} optimizer steps')

## 🚀 Run Training & Resume From Checkpoints

This section starts the actual fine-tuning process. It prepares the optimizer, scheduler, and trainer, then checks whether previous training progress exists so training can continue instead of starting from zero.

---

### 📁 Checkpoint Directory

Local checkpoints are stored in:

```python
./checkpoints
```

However, Kaggle’s working directory is temporary. When the session ends, local files may be deleted.

So local checkpoints are useful only during the same session.

---

### ☁️ Durable Hugging Face Checkpoint Storage

The model checkpoints are also pushed to a Hugging Face model repository:

```python
AgentPhoenix7/SLM-project-qwen
```

This is important because Hugging Face Hub survives Kaggle session resets.

So the training can continue later from:

```python
checkpoints/last.pt
```

---

### 🔢 Total Steps and Warmup Steps

The code calculates:

* Total optimizer steps across all epochs
* Number of warmup steps based on the warmup ratio

Warmup means the learning rate starts small and gradually increases before normal scheduling begins.

This avoids unstable training at the beginning.

---

### ⚙️ Optimizer and Scheduler Creation

The optimizer controls how model parameters are updated.

The scheduler controls how the learning rate changes over time.

This cell creates:

* `AdamW` optimizer
* Cosine learning-rate scheduler
* Custom `Trainer` object

The trainer receives:

* Model
* Train loader
* Validation loader
* Optimizer
* Scheduler
* Device
* Checkpoint path
* Hugging Face repo ID

At this point, everything required for training is assembled.

---

### 🔄 Resume Logic

Before training starts, the notebook checks for existing checkpoints.

The priority order is:

1. **Hugging Face Hub checkpoint**

   * Best option because it survives session resets

2. **Local `last.pt` checkpoint**

   * Useful if the notebook kernel restarted in the same Kaggle session

3. **Latest local epoch checkpoint**

   * Fallback if `last.pt` is unavailable

4. **No checkpoint found**

   * Training starts from scratch

This prevents losing progress after interruptions.

---

### 🧠 Actual Training

Finally, this line starts training:

```python
best_loss = trainer.train(NUM_EPOCHS, start_epoch=start_epoch)
```

The trainer:

* Runs training batches
* Computes loss
* Updates LoRA parameters
* Runs validation
* Saves checkpoints
* Pushes checkpoints to Hugging Face
* Tracks the best validation loss

---

### ✅ Outcome of This Cell

After this step:

* Training starts or resumes automatically
* Checkpoints are loaded if available
* Model progress is saved locally and remotely
* Best validation loss is printed at the end

Here the model actually learns from the Amazon review dataset.


In [ ]:
# ── Cell 6: run training ──────────────────────────────────────────────────────
CKPT_DIR = './checkpoints'

# ── REQUIRED for cross-session resume ─────────────────────────────────────────
# /kaggle/working is wiped when a session ends. The HF Hub repo below is the
# only durable store. Set HF_REPO_ID to your repo; it is created automatically
# as private if it does not exist. Re-running all cells resumes automatically.
HF_REPO_ID = 'AgentPhoenix7/SLM-project-qwen'

if not HF_REPO_ID:
    print('WARNING: HF_REPO_ID not set — checkpoints will be lost on session end.')

TOTAL_STEPS  = NUM_EPOCHS * (len(train_loader) // ACCUMULATION_STEPS)
WARMUP_STEPS = int(WARMUP_RATIO * TOTAL_STEPS)

optimizer = configure_optimizer(model, lr=MAX_LR, weight_decay=WEIGHT_DECAY)
scheduler = CosineScheduler(optimizer, WARMUP_STEPS, TOTAL_STEPS, MAX_LR, MIN_LR)
trainer   = Trainer(
    model,
    train_loader,
    val_loader,
    scheduler,
    optimizer=optimizer,
    device=DEVICE,
    use_amp=True,
    accumulation_steps=ACCUMULATION_STEPS,
    checkpoint_dir=CKPT_DIR,
    hf_repo_id=HF_REPO_ID,
)

# ── resume logic (same pattern as bert-finetune-kaggle.ipynb) ─────────────────
import glob
start_epoch = 0
resumed     = False

# 1. Try HF Hub first — only store that survives session end
if HF_REPO_ID and not resumed:
    try:
        from huggingface_hub import hf_hub_download
        hub_path = hf_hub_download(
            repo_id=HF_REPO_ID,
            filename='checkpoints/last.pt',
            repo_type='model',
            force_download=True,
        )
        start_epoch = trainer.load_checkpoint(hub_path)
        print(f'Resumed from HF Hub ({HF_REPO_ID}), start_epoch={start_epoch}')
        resumed = True
    except Exception as e:
        print(f'HF Hub: no checkpoint found ({e})')

# 2. Fall back to local last.pt (same-session kernel restart)
if not resumed:
    local_last = os.path.join(CKPT_DIR, 'last.pt')
    if os.path.exists(local_last):
        start_epoch = trainer.load_checkpoint(local_last)
        print(f'Resumed from local last.pt, start_epoch={start_epoch}')
        resumed = True

# 3. Fall back to latest epoch checkpoint
if not resumed:
    epoch_ckpts = sorted(glob.glob(os.path.join(CKPT_DIR, 'epoch_*.pt')))
    if epoch_ckpts:
        start_epoch = trainer.load_checkpoint(epoch_ckpts[-1])
        print(f'Resumed from {epoch_ckpts[-1]}, start_epoch={start_epoch}')
        resumed = True

if not resumed:
    print('No checkpoint found — starting fresh.')

best_loss = trainer.train(NUM_EPOCHS, start_epoch=start_epoch)
print('Best val loss:', best_loss)

## 📊 Final Evaluation & Confusion Matrix

This section evaluates the trained model on the validation dataset and visualizes its performance using a confusion matrix.

---

### 📁 Loading the Best Checkpoint

The code first looks for:

```python
best.pt
```

This checkpoint represents the model with the lowest validation loss during training.

If `best.pt` is not available locally, the notebook tries to download it from Hugging Face Hub.

If that also fails, it falls back to:

```python
last.pt
```

This ensures evaluation can still run even if the best checkpoint is missing.

---

### 🧠 Restoring the Trained Model

The selected checkpoint is loaded into the trainer:

```python
trainer.load_checkpoint(best_path)
```

Then the model is moved to the correct device and switched to evaluation mode.

Evaluation mode disables training-specific behavior such as dropout, making predictions stable and consistent.

---

### 🔍 Running Predictions

The model processes the validation dataset batch by batch.

For each batch:

1. Input IDs and attention masks are moved to GPU/CPU
2. The model produces logits
3. The class with the highest logit is selected as the prediction
4. Predictions and true labels are stored

The output class labels are:

```text
0, 1, 2, 3, 4
```

which correspond to:

```text
1-star, 2-star, 3-star, 4-star, 5-star
```

---

### 🧾 Confusion Matrix

A confusion matrix compares:

* Actual labels
* Predicted labels

It shows where the model is correct and where it gets confused.

For example:

* High values on the diagonal mean correct predictions
* Off-diagonal values show mistakes

This is useful for seeing whether the model confuses nearby ratings, such as 4-star vs 5-star.

---

### 📈 Saving the Plot

The confusion matrix is saved as:

```python
confusion_matrix.png
```

inside the checkpoint directory.

This gives a reusable visual result for reports, presentations, or project documentation.

---

### ✅ Evaluation Metrics

Two metrics are printed:

#### Accuracy

Accuracy measures how often the model predicts the exact correct rating.

Example:

```text
Actual: 5-star
Predicted: 5-star
```

That counts as correct.

---

#### MAE — Mean Absolute Error

MAE measures how far the predicted rating is from the true rating.

Example:

```text
Actual: 5-star
Predicted: 4-star
Error = 1
```

MAE is especially useful here because star ratings are ordered.

Predicting 4 instead of 5 is not as bad as predicting 1 instead of 5.

---

### 🎯 Why This Step Matters

Training loss alone is not enough. This cell shows whether the model actually performs well on unseen validation data.

It answers:

* How accurate is the model?
* Which ratings does it confuse?
* Is the model making small mistakes or large mistakes?
* Did the fine-tuning actually improve performance?

---

### ✅ Outcome of This Cell

After this step:

* Best available checkpoint is loaded
* Validation predictions are generated
* Confusion matrix is displayed and saved
* Accuracy and MAE are reported

This is the final result-checking stage of the fine-tuning pipeline.


In [ ]:
# ── Cell 7: results / confusion matrix ───────────────────────────────────────

import os
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score, mean_absolute_error

best_path = os.path.join(CKPT_DIR, "best.pt")
last_path = os.path.join(CKPT_DIR, "last.pt")

if not os.path.exists(best_path) and HF_REPO_ID:
    try:
        from huggingface_hub import hf_hub_download
        best_path = hf_hub_download(
            repo_id=HF_REPO_ID,
            filename="checkpoints/best.pt",
            repo_type="model",
            force_download=True,
        )
        print("Loaded best.pt from Hugging Face Hub")
    except Exception:
        print("best.pt not found on Hub, using last.pt instead")
        best_path = last_path

if not os.path.exists(best_path):
    best_path = last_path

trainer.load_checkpoint(best_path)

model.to(DEVICE)
model.eval()

all_preds, all_labels = [], []
total_batches = len(val_loader)

with torch.no_grad():
    for batch_idx, batch in enumerate(val_loader):
        ids = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)

        outputs = model(ids, attention_mask=mask)

        if hasattr(outputs, "logits"):
            logits = outputs.logits
        elif isinstance(outputs, tuple):
            logits = outputs[0]
        else:
            logits = outputs

        preds = logits.argmax(-1).cpu().tolist()

        all_preds += preds
        all_labels += batch["label"].tolist()

        if (batch_idx + 1) % 100 == 0:
            print(f"Evaluated {batch_idx + 1}/{total_batches} batches")

cm = confusion_matrix(all_labels, all_preds, labels=[0, 1, 2, 3, 4])
disp = ConfusionMatrixDisplay(cm, display_labels=[1, 2, 3, 4, 5])
disp.plot(cmap="Blues")

plt.title("Qwen2.5-0.5B Amazon Review — Confusion Matrix")
plt.tight_layout()
plt.savefig(os.path.join(CKPT_DIR, "confusion_matrix.png"), dpi=150)
plt.show()

acc = accuracy_score(all_labels, all_preds)
mae = mean_absolute_error(all_labels, all_preds)

print(f"Accuracy : {acc:.4f}")
print(f"MAE      : {mae:.4f}")

## 🧾 Conclusion

In this project, a pretrained transformer model — Qwen2.5-0.5B — was successfully adapted for multi-class sentiment classification on Amazon book reviews using **LoRA**.

Instead of performing full fine-tuning (which is computationally expensive), LoRA enabled efficient training by updating only a small subset of parameters within the attention layers, while keeping the base model largely frozen. This significantly reduced memory usage and training time without sacrificing performance.

The pipeline was designed to be robust and production-ready, incorporating:

* Efficient dataset handling with streaming and caching
* Gradient accumulation for larger effective batch sizes
* Cosine learning rate scheduling with warmup for stable convergence
* Mixed precision (bfloat16) for faster computation
* Automatic checkpointing and Hugging Face Hub integration for fault tolerance and cross-session recovery

The trained model demonstrated its performance through validation metrics such as accuracy and mean absolute error, along with a confusion matrix that provided deeper insight into prediction behavior across different rating classes.

Overall, this workflow highlights a practical and scalable approach to adapting large language models for real-world classification tasks, balancing efficiency, performance, and reliability.
